In [1]:
# ch04_HUMAN_IN_THE_LOOP.ipynb

In [2]:
from dotenv import load_dotenv
load_dotenv()

True

In [9]:
from langgraph.types import interrupt
from langchain_core.tools import tool
from langchain.chat_models import init_chat_model
from langchain_tavily import TavilySearch

llm = init_chat_model("openai:gpt-4.1")

@tool
def human_assistance(query: str):
    """사람의 도움이 필요할 때 호출되는 도구입니다."""
    human_response = interrupt({"query": query})  # 실행 일시 중단
    return human_response

/Users/gyungah/.pyenv/versions/py313/lib/python3.13/site-packages/langchain_tavily/tavily_research.py:97: UserWarning: Field name "output_schema" in "TavilyResearch" shadows an attribute in parent "BaseTool"
  class TavilyResearch(BaseTool):  # type: ignore[override, override]
/Users/gyungah/.pyenv/versions/py313/lib/python3.13/site-packages/langchain_tavily/tavily_research.py:97: UserWarning: Field name "stream" in "TavilyResearch" shadows an attribute in parent "BaseTool"
  class TavilyResearch(BaseTool):  # type: ignore[override, override]


In [10]:
search_tool = TavilySearch(max_results=2)
tools = [search_tool, human_assistance]
llm_with_tools = llm.bind_tools(tools)

In [16]:
def chatbot(state: State):
    message = llm_with_tools.invoke(state["messages"])
    return {"messages": [message]}

In [17]:
from langgraph.graph import StateGraph, START, END
from langgraph.prebuilt import ToolNode, tools_condition
from langgraph.checkpoint.memory import MemorySaver
from typing import Annotated
from typing_extensions import TypedDict
from langgraph.graph.message import add_messages

# 상태 정의
class State(TypedDict):
    messages: Annotated[list, add_messages]

# 그래프 초기화 및 메모리 설정
graph_builder = StateGraph(State)
memory = MemorySaver()

# 노드 및 엣지 추가
graph_builder.add_node("chatbot", chatbot)
tool_node = ToolNode(tools=tools)
graph_builder.add_node("tools", tool_node)
graph_builder.add_conditional_edges("chatbot", tools_condition)
graph_builder.add_edge("tools", "chatbot")
graph_builder.add_edge(START, "chatbot")

# 그래프 컴파일
graph = graph_builder.compile(checkpointer=memory)

In [18]:
user_input = "AI 에이전트를 만드는 데 전문가의 조언이 필요해요. 사람의 도움을 받을 수 있을까요?"
config = {"configurable": {"thread_id": "1"}}

events = graph.stream(
    {"messages": [{"role": "user", "content": user_input}]},
    config,
    stream_mode="values",
)
for event in events:
    if "messages" in event:
        event["messages"][-1].pretty_print()

================================ Human Message =================================

AI 에이전트를 만드는 데 전문가의 조언이 필요해요. 사람의 도움을 받을 수 있을까요?
================================== Ai Message ==================================
Tool Calls:
  human_assistance (call_8GUlDnNsMC7iBefe9ulg4fjv)
 Call ID: call_8GUlDnNsMC7iBefe9ulg4fjv
  Args:
    query: AI 에이전트 개발에 관한 전문가의 조언이 필요합니다. 사람의 도움을 연결해 주세요.
================================== Ai Message ==================================
Tool Calls:
  human_assistance (call_8GUlDnNsMC7iBefe9ulg4fjv)
 Call ID: call_8GUlDnNsMC7iBefe9ulg4fjv
  Args:
    query: AI 에이전트 개발에 관한 전문가의 조언이 필요합니다. 사람의 도움을 연결해 주세요.


In [19]:
snapshot = graph.get_state(config)
print(snapshot.next)  # 출력: ('tools',)

('tools',)


In [20]:
from langgraph.types import Command

human_response = (
    "전문가입니다. LangGraph를 활용해 AI 에이전트를 만들어보세요. "
    "간단한 자율 에이전트보다 훨씬 더 확장성 있고 안정적입니다."
)

human_command = Command(resume={"data": human_response})

events = graph.stream(human_command, config, stream_mode="values")
for event in events:
    if "messages" in event:
        event["messages"][-1].pretty_print()

================================== Ai Message ==================================
Tool Calls:
  human_assistance (call_8GUlDnNsMC7iBefe9ulg4fjv)
 Call ID: call_8GUlDnNsMC7iBefe9ulg4fjv
  Args:
    query: AI 에이전트 개발에 관한 전문가의 조언이 필요합니다. 사람의 도움을 연결해 주세요.
================================= Tool Message =================================
Name: human_assistance

{"data": "전문가입니다. LangGraph를 활용해 AI 에이전트를 만들어보세요. 간단한 자율 에이전트보다 훨씬 더 확장성 있고 안정적입니다."}
================================== Ai Message ==================================

전문가의 답변입니다:

AI 에이전트를 만들 때 LangGraph를 활용하는 것을 추천합니다. LangGraph는 단순한 자율 에이전트보다 훨씬 더 확장성과 안정성을 제공합니다. 

LangGraph는 다양한 작업 흐름을 구성하고, 여러 에이전트 또는 기능을 단계별로 조합할 수 있도록 해줘서 복잡한 AI 시스템을 설계할 때 매우 유용합니다.

특정한 구현이나 사용법에 대해 더 궁금한 점이 있으신가요? 아니면 어떤 종류의 AI 에이전트를 만들고 싶으신지 좀 더 구체적으로 말씀해주시면, 맞춤 조언을 드릴 수 있습니다!
